# Experiment Tracker - Resumen actualizado

Este notebook documenta la trazabilidad de los experimentos del proyecto y actualiza la seleccion final luego de comparar Random Forest, XGBoost y LightGBM sobre el mismo test real desbalanceado.

La conclusion ya no es un unico modelo ganador universal. La seleccion depende del escenario de negocio: costo de contacto, necesidad de capturar compradores y tipo de visitante.


## Artefactos usados

- `models/experiments_log.csv`: resultados baseline del Sprint 3.
- `models/experiments_log_sprint4.csv`: resumen historico del Sprint 4.
- `models/ensemble_results_xgb_lgbm.csv`: comparacion de XGBoost, LightGBM y Random Forest en test real.
- `models/ensemble_results_by_visitor_type.csv`: evaluacion segmentada por `VisitorType` desde el notebook 13.
- `models/final_test_model_comparison.csv`: comparacion final usada por el notebook 14.
- `models/final_test_by_visitor_type.csv`: validacion final segmentada usada por el notebook 14.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

MODELS_DIR = ROOT / "models"

paths = {
    "baseline_sprint3": MODELS_DIR / "experiments_log.csv",
    "sprint4_summary": MODELS_DIR / "experiments_log_sprint4.csv",
    "ensemble_test": MODELS_DIR / "ensemble_results_xgb_lgbm.csv",
    "ensemble_by_visitor_type": MODELS_DIR / "ensemble_results_by_visitor_type.csv",
    "final_test": MODELS_DIR / "final_test_model_comparison.csv",
    "final_by_visitor_type": MODELS_DIR / "final_test_by_visitor_type.csv",
}

for name, path in paths.items():
    print(f"{name}: {'OK' if path.exists() else 'NO ENCONTRADO'} - {path}")


## Comparacion final en test real

El criterio principal para la comparacion final es el desempeno en `test_processed.csv`, que conserva la proporcion real de compras cercana al 15%.

Se reportan dos politicas:

- `threshold=0.50`: escenario conservador, prioriza precision y eficiencia de contacto.
- `threshold=0.25`: escenario agresivo, prioriza capturar mas compradores.


In [ ]:
final_results = pd.read_csv(paths["final_test"])

cols = [
    "model",
    "version",
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]

display(final_results[cols])


## Seleccion por escenario

La seleccion final queda definida asi:

| Escenario | Modelo recomendado | Threshold | Razon |
|---|---|---:|---|
| Global conservador / ranking | XGBoost tuned | 0.50 | Mayor PR-AUC y ROC-AUC en test real; util cuando contactar usuarios tiene costo. |
| Maxima captura de compradores | Random Forest | 0.25 | Mayor recall global; util si perder compradores es mas caro que contactar falsos positivos. |
| Usuarios nuevos | XGBoost base/tuned | 0.50 | Alta precision; contactar nuevos suele ser mas dificil y caro. |
| Usuarios recurrentes | Random Forest | 0.25 | Alto recall; activar recurrentes suele ser mas barato. |

Por eso, el tracker deja de registrar a Random Forest como unico ganador universal. Random Forest sigue siendo valioso para captura, pero XGBoost es mejor para ranking/campanas conservadoras.


In [ ]:
segment_results = pd.read_csv(paths["final_by_visitor_type"])

segment_cols = [
    "visitor_type",
    "model",
    "version",
    "threshold",
    "segment_size",
    "segment_buyers",
    "segment_conversion_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]

display(segment_results[segment_cols])


## Interpretacion ejecutiva

- El mejor modelo global para ranking y campanas con costo de contacto es **XGBoost tuned**.
- El mejor modelo para capturar la mayor cantidad posible de compradores es **Random Forest con threshold 0.25**.
- Para `New_Visitor`, conviene una politica conservadora por precision.
- Para `Returning_Visitor`, puede aceptarse un threshold mas bajo para maximizar recall.
- `LightGBM` queda como modelo competitivo, pero no supera claramente a XGBoost ni a Random Forest en los escenarios clave.

Esta trazabilidad conecta las metricas tecnicas con la decision de negocio: no se despliega un unico clasificador de forma ciega, sino una politica por escenario.
